<a href="https://colab.research.google.com/github/evildead23151/3D-Projects/blob/main/Project_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install open3d plotly

In [4]:
import numpy as np
import open3d as o3d
import plotly.graph_objects as go

# =========================
# 1. Sonar parameters
# =========================
num_rays = 300
angles = np.linspace(-np.pi / 4, np.pi / 4, num_rays)

beam_width = np.deg2rad(1.5)     # beam spread
beam_samples = 5                # samples per beam

range_noise_std = 0.05           # meters
angle_noise_std = np.deg2rad(0.2)

# =========================
# 2. Cylinder definition
# =========================
cylinder_radius = 3.0
cylinder_center = np.array([6.0, 0.0])

# =========================
# 3. Ray-cylinder intersection
# =========================
def intersect_ray_cylinder(theta):
    dx = np.cos(theta)
    dy = np.sin(theta)

    # Ray: (t*dx, t*dy)
    # Cylinder: (x-cx)^2 + (y-cy)^2 = r^2

    cx, cy = cylinder_center

    a = dx*dx + dy*dy
    b = 2 * (dx*(-cx) + dy*(-cy))
    c = cx*cx + cy*cy - cylinder_radius**2

    disc = b*b - 4*a*c
    if disc < 0:
        return None

    t1 = (-b - np.sqrt(disc)) / (2*a)
    t2 = (-b + np.sqrt(disc)) / (2*a)

    t = min(t for t in [t1, t2] if t > 0) if any(t > 0 for t in [t1, t2]) else None
    return t

# =========================
# 4. Generate sonar points
# =========================
points = []

for theta in angles:

    # Angular noise
    noisy_theta = theta + np.random.normal(0, angle_noise_std)

    t_hit = intersect_ray_cylinder(noisy_theta)
    if t_hit is None:
        continue

    for _ in range(beam_samples):

        # Beam spread
        beam_theta = noisy_theta + np.random.uniform(-beam_width/2, beam_width/2)

        dx = np.cos(beam_theta)
        dy = np.sin(beam_theta)

        # Range noise
        noisy_range = t_hit + np.random.normal(0, range_noise_std)

        x = noisy_range * dx
        y = noisy_range * dy
        z = np.random.normal(0, 0.02)   # small vertical jitter

        points.append([x, y, z])

points = np.array(points)

# =========================
# 5. Open3D Point Cloud
# =========================
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(points)

# =========================
# 6. Headless visualization
# =========================
fig = go.Figure(
    data=[
        go.Scatter3d(
            x=points[:,0],
            y=points[:,1],
            z=points[:,2],
            mode='markers',
            marker=dict(size=2)
        )
    ]
)

fig.update_layout(
    title="Sonar Fan → Noisy Cylinder Point Cloud",
    scene=dict(
        xaxis_title="X",
        yaxis_title="Y",
        zaxis_title="Z",
        aspectmode="data"
    )
)

fig.show()
